<h2> Tokenizer <h2\>

In [1]:
import torch
import pandas as pd

class CharTokenizer:
    def __init__(self, text=None, pad_token='<PAD>', unk_token='<UNK>'):
        self.pad_token = pad_token
        self.unk_token = unk_token
        self.char2idx = {}
        self.idx2char = {}
        self.vocab_built = False

        if text:
            self.build_vocab(text)
        
        self.vocab_size = len(self.char2idx)

    def build_vocab(self, text):
        unique_chars = sorted(set(text))
        # Reserve indices for PAD and UNK
        self.char2idx = {self.pad_token: 0, self.unk_token: 1}
        for i, ch in enumerate(unique_chars, start=2):
            self.char2idx[ch] = i
        self.idx2char = {i: ch for ch, i in self.char2idx.items()}
        self.vocab_built = True

    def encode(self, text, max_length=None):
        if not self.vocab_built:
            raise ValueError("Vocabulary not built yet. Call build_vocab first.")

        encoded = [self.char2idx.get(ch, self.char2idx[self.unk_token]) for ch in text]

        # Truncate if needed
        if max_length is not None:
            encoded = encoded[:max_length]

        # Pad if needed
        if max_length is not None and len(encoded) < max_length:
            pad_length = max_length - len(encoded)
            encoded += [self.char2idx[self.pad_token]] * pad_length

        return torch.tensor(encoded, dtype=torch.long)

    def decode(self, indices):
        # Accept either tensor or list
        if isinstance(indices, torch.Tensor):
            indices = indices.tolist()
        chars = [self.idx2char.get(i, self.unk_token) for i in indices]
        # Strip padding tokens from end
        while chars and chars[-1] == self.pad_token:
            chars.pop()
        return ''.join(chars)
    
    def create_mask(self, encoded_tensor):
        pad_token_idx = self.char2idx[self.pad_token]
        return (encoded_tensor != pad_token_idx).long()


<h2> DataSet <h2\>

In [2]:
from torch.utils.data import Dataset

class CISC2010DataSet(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=1024):
        self.df = dataframe.drop_duplicates(subset='content', keep='first')
        self.df = self.df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.loc[idx]
        text = row["content"]
        label = int(row["classification"])

        encoded = self.tokenizer.encode(text, max_length=self.max_len)
        mask = self.tokenizer.create_mask(encoded)

        return encoded, torch.tensor(label, dtype=torch.long)


<h2> The Model <h2\>

In [3]:
import torch.nn as nn
import torch.nn.functional as F


class PLNModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, max_length, anchor_sizes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.max_length = max_length
        self.anchor_sizes = anchor_sizes
        self.num_anchors = len(anchor_sizes)

        # Simple feature extractor
        self.feature_extractor = nn.Sequential(
            nn.Conv1d(embedding_dim, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.ReLU()
        )

        self.cls_head = nn.Conv1d(128, self.num_anchors * 2, kernel_size=1)


    def forward(self, x):
        """
        Args:
            x (Tensor): Input token indices, shape (B, Lmax)

        Returns:
            cls_logits: (B, L * num_anchors, 2)
        """
        # 1. Embed input (B, L) -> (B, L, embedding_dim)
        x_emb = self.embedding(x)

        # 2. Convert to (B, embedding_dim, L) for Conv1d
        x_emb = x_emb.permute(0, 2, 1)

        # 3. Feature extractor: (B, embedding_dim, L) -> (B, 128, L)
        features = self.feature_extractor(x_emb)

        # 4. classification 
        cls_logits = self.cls_head(features)

        # 5. Reshape outputs:
        B, _, L = cls_logits.shape
        # cls_logits shape (B, 2 * num_anchors, L) -> (B, L * num_anchors, 2)
        cls_logits = cls_logits.view(B, self.num_anchors, 2, L)
        cls_logits = cls_logits.permute(0, 3, 1, 2).contiguous()
        cls_logits = cls_logits.view(B, L * self.num_anchors, 2)

        return cls_logits


<h2> train function <h2\>

In [4]:
import os
from tqdm.auto import tqdm


def train_pln(model,
              dataloader,
              optimizer,
              loss_function,
              device,
              num_anchors,
              epochs: int = 10,
              save_every: int = 5000,
              save_dir: str = "checkpoints"):

    os.makedirs(save_dir, exist_ok=True)
    model.to(device)
    
    start_epoch = 1
    ckpts = [f for f in os.listdir(save_dir) if f.endswith(".pt")]
    if ckpts:
        latest_ckpt = max(ckpts, key=lambda x: int(x.split("_")[-1].split(".")[0]))
        ckpt_path = os.path.join(save_dir, latest_ckpt)
        checkpoint = torch.load(ckpt_path, map_location=device)
        
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        start_epoch = checkpoint["epoch"] + 1
        
        tqdm.write(f"✅ Loaded checkpoint '{ckpt_path}' (epoch {checkpoint['epoch']})")
    

    epoch_bar = tqdm(range(start_epoch, epochs + 1), desc="Epochs", unit="epoch", leave=False)

    for epoch in epoch_bar:
        model.train()
        running_loss = 0.0

        for inputs, labels in dataloader:
            inputs = inputs.to(device)           # (B, L)
            labels = labels.to(device).long()    # (B,)

            optimizer.zero_grad()
            cls_logits = model(inputs)           # (B, L*A, 2)
            loss = loss_function(cls_logits, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        avg_loss = running_loss / len(dataloader)
        epoch_bar.set_postfix(avg_loss=f"{avg_loss:.4f}")

        # ── checkpoint every `save_every` epochs ───────────────────────────
        if epoch % save_every == 0:
            ckpt_path = os.path.join(save_dir, f"pln_epoch_{epoch}.pt")
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                },
                ckpt_path,
            )
            tqdm.write(f"✓ Saved checkpoint → {ckpt_path}")

<h2> loss function <h2\>

In [5]:
def request_loss(cls_logits, request_targets):

    # Pool over all anchors in the request – pick the most suspicious one.
    request_logits = cls_logits.max(dim=1).values   # (B, 2)

    # Standard CE on the pooled logits.
    loss = F.cross_entropy(request_logits, request_targets, reduction='mean')
    return loss

<h2> eval function <h2\>

In [6]:
from sklearn.metrics import precision_score, recall_score, f1_score
from tqdm import tqdm
import torch.nn.functional as F

def evaluate(model, dataloader, device, tokenizer, max_samples):
    model.eval()
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    correct_samples = []
    incorrect_samples = []

    batch_bar = tqdm(enumerate(dataloader), total=len(dataloader), unit="batch", leave=False)
    with torch.no_grad():
        for batch_idx, (inputs, labels) in batch_bar:
            inputs = inputs.to(device)
            labels = labels.to(device)
        
            batch_bar.set_description(f"Batch {batch_idx}")

            logits = model(inputs)
            request_logits = logits.max(dim=1).values
            preds = request_logits.argmax(dim=-1)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

            correct_mask = (preds == labels)
            correct += correct_mask.sum().item()
            total += labels.size(0)

            for i in range(len(labels)):
                if len(correct_samples) >= max_samples and len(incorrect_samples) >= max_samples:
                    break
                decoded_input = tokenizer.decode(inputs[i].cpu())
                true_label = labels[i].item()
                pred_label = preds[i].item()

                sample = {
                    'input': decoded_input,
                    'true_label': true_label,
                    'pred_label': pred_label,
                }

                if pred_label == true_label and len(correct_samples) < max_samples:
                    correct_samples.append(sample)
                elif pred_label != true_label and len(incorrect_samples) < max_samples:
                    incorrect_samples.append(sample)

            if len(correct_samples) >= max_samples and len(incorrect_samples) >= max_samples:
                break

    accuracy = correct / total if total > 0 else 0
    precision = precision_score(all_labels, all_preds, zero_division=0)
    recall = recall_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)


    return accuracy, precision, recall, f1, correct_samples, incorrect_samples

<h1> All comes together <h1\>

In [ ]:
from torch.utils.data import DataLoader
import torch.optim as optim

def main():

    max_len = 200

    # Load dataset
    train_csv_path = "CISC2010_cleaned_train.csv"
    train_df = pd.read_csv(train_csv_path)

    test_csv_path = "CISC2010_cleaned_train.csv"
    test_df = pd.read_csv(test_csv_path)

    # Build tokenizer vocab from all content
    tokenizer = CharTokenizer("".join(train_df["content"].tolist()))

    # Dataset + DataLoader
    train_dataset = CISC2010DataSet(train_df, tokenizer, max_len=max_len)
    train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    test_dataset = CISC2010DataSet(test_df, tokenizer, max_len=max_len)
    test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"running on {device}")

    # Init model
    vocab_size = tokenizer.vocab_size
    embedding_dim = 64
    anchor_sizes = [4, 8, 16]
    model = PLNModel(vocab_size, embedding_dim, max_len, anchor_sizes)

    # Optimizer
    print(f"{sum(p.numel() for p in model.parameters() if p.requires_grad)} parameters to optimaize")
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    # Train
    train_pln(
        model = model, 
        dataloader = train_dataloader, 
        optimizer = optimizer, 
        device = device,
        num_anchors = len(anchor_sizes),
        loss_function = request_loss,
        epochs=400, 
        save_every=50,
        save_dir = "checkpoints"
        )

    
    accuracy, precision, recall, f1, correct_samples, incorrect_samples = evaluate(
        model=model,
        dataloader=test_dataloader,
        device=device,
        tokenizer=tokenizer,
        max_samples=10
        )
    
    #print(model(tokenizer.encode("http://localhost:8080/tienda1/publico/anadir.jsp HTTP/1.1 id=1&nombre=Jam%F3n+Ib%E9rico&precio=39&cantidad=3&B1=A%F1adir+al+carrito").to(device).unsqueeze(0)).max(dim=1).values.argmax(dim=-1))
    
    print("\n====== Final Evaluation Metrics ======")
    print(f"Accuracy   : {accuracy:.4f}")
    print(f"Precision  : {precision:.4f}")
    print(f"Recall     : {recall:.4f}")
    print(f"F1 Score   : {f1:.4f}")
    print("======================================\n")

    print("Sample Correct Predictions:")
    for sample in correct_samples:
        print(f"Input      : {sample['input']}")
        print(f"True Label : {sample['true_label']}, Predicted: {sample['pred_label']}")
        print("-----")

    print("\nSample Incorrect Predictions:")
    for sample in incorrect_samples:
        print(f"Input      : {sample['input']}")
        print(f"True Label : {sample['true_label']}, Predicted: {sample['pred_label']}")
        print("-----")

if __name__ == "__main__":
    main()

running on cuda
79686 parameters to optimaize
✅ Loaded checkpoint 'checkpoints/pln_epoch_400.pt' (epoch 400)


tensor([0], device='cuda:0')

====== Final Evaluation Metrics ======
Accuracy   : 0.9323
Precision  : 1.0000
Recall     : 0.8839
F1 Score   : 0.9384

Sample Correct Predictions:
Input      : http://localhost:8080/tienda1/publico/entrar.jsp HTTP/1.1 errorMsg=Credenciales+incorrectas%27%3B+DROP+TABLE+usuarios%3B+SELECT+*+FROM+datos+WHERE+nombre+LIKE+%27%25
True Label : 1, Predicted: 1
-----
Input      : http://localhost:8080/tienda1/publico/productos.jsp HTTP/1.1 
True Label : 0, Predicted: 0
-----
Input      : http://localhost:8080/tienda1/miembros/editar.jsp HTTP/1.1 modo=registro&login=gantt&password=sae19r8&nombre=Libe&apellidos=Amusquivar&email=mccormack%40neotelecom.tn&dni=35453403F&direccion=Calle+Tam
True Label : 1, Predicted: 1
-----
Input      : http://localhost:8080/tienda1/global/titulo.jsp HTTP/1.1 
True Label : 0, Predicted: 0
-----
Input      : http://localhost:8080/tienda1/publico/pagar.jsp?modo=insertar&precio=5628&B1=Pasar+por+caja HTTP/1.1 
True Label : 0, Predicted: 0